# Arbeitspaket (AP) 1: Management & Nutzung relationaler Daten

### Persönliche Angaben (bitte ergänzen) und Prüfungsregeln

<table>
  <tr><td>Vorname:</td><td></td></tr>
  <tr><td>Nachname:</td><td></td></tr>
  <tr><td>Immatrikulationsnummer:</td><td></td></tr>
  <tr><td>Modul:</td><td>Data Science</td></tr>
  <tr><td>Prüfungsdatum / Raum / Zeit:</td><td>[TT.MM.2026] / Raum: [anpassen] / [hh:mm – hh:mm]</td></tr>
  <tr><td>Erlaubte Hilfsmittel:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Nicht erlaubt:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Technische Probleme:</td><td>Wenden Sie sich bei technischen Problemen umgehend an die Prüfungsaufsicht.</td></tr>
</table>

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

<table>
  <thead>
    <tr><th>Teil</th><th>Thema</th><th>Aufgaben</th><th>Punkte</th></tr>
  </thead>
  <tbody>
    <tr><td>A</td><td>Konzeptfragen: Datenmodellierung &amp; Design</td><td>A1–A3</td><td>12</td></tr>
    <tr><td>B</td><td>SQL-Fehleranalyse (cinema.db)</td><td>B1–B2</td><td>16</td></tr>
    <tr><td>C</td><td>SQL-Abfragen schreiben (cinema.db)</td><td>C1–C3</td><td>12</td></tr>
    <tr><td><b>Total</b></td><td></td><td><b>8 Aufgaben</b></td><td><b>40</b></td></tr>
  </tbody>
</table>

### Bewertungsschema

Jede Aufgabe wird für sich und **als Ganzes** bewertet. Ihre Punktzahl steht in der Überschrift.

<table>
  <thead>
    <tr><th>Kategorie</th><th>Beschreibung</th><th>Bewertung</th><th>4 Pkt</th><th>8 Pkt</th></tr>
  </thead>
  <tbody>
    <tr><td>Korrekt</td><td>Vollständig und fachlich einwandfrei: Antwort bzw. Begründung korrekt und nachvollziehbar; bei SQL-Aufgaben liefert die SQL-Abfrage das richtige Ergebnis</td><td>100%</td><td>4</td><td>8</td></tr>
    <tr><td>Minimale Mängel</td><td>Kleine Fehler (z.B. falsche Sortierung, fehlende Rundung) oder Begründung ungenau bzw. unvollständig</td><td>75%</td><td>3</td><td>6</td></tr>
    <tr><td>Mittlere Mängel</td><td>Grössere Fehler (z.B. fehlende Spalten, falsche Filterung) oder Begründung fehlt bzw. ist fachlich falsch</td><td>50%</td><td>2</td><td>4</td></tr>
    <tr><td>Gravierende Mängel</td><td>Problem bzw. Ansatz erkennbar, aber Lösung oder Schlussfolgerung im Kern falsch (z.B. Problem richtig benannt, aber falsch behoben; im Code: ungeeigneter Lösungsweg wie falscher JOIN-Typ oder fehlende Tabelle)</td><td>25%</td><td>1</td><td>2</td></tr>
    <tr><td>Nicht lauffähig / nicht sinnvoll</td><td>Antwort fehlt, Code nicht lauffähig (z.B. Syntaxfehler) oder kein Bezug zur Aufgabe</td><td>0%</td><td>0</td><td>0</td></tr>
  </tbody>
</table>

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege werden ebenfalls als korrekt gewertet.

## Hinweise zur Datenbank

**Datenbank:** PostgreSQL — Datenbank `postgres`, Benutzer `pgadmin`. Die Verbindung wird im Setup-Abschnitt aufgebaut und getestet.

**Backup:** `cinema.db` (SQLite) im gleichen Verzeichnis wie dieses Notebook — nur zu verwenden, falls die PostgreSQL-Verbindung nicht funktioniert.

**Tabellen:** `movies`, `cinemas`, `staff`, `screenings`, `customers`, `tickets`

Das ER-Diagramm und die Tabellenbeschreibungen finden Sie im Sheet **«ER-Diagram»** bzw. **«Description»** der beiliegenden Datei `cinema_data.xlsx`.

**Antworten auf Konzeptfragen** tragen Sie bitte direkt in die vorgesehenen Markdown-Zellen ein.

## Setup (wird nicht bewertet)

In [1]:
# Libraries
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')

print('Working directory:', os.getcwd())

Working directory: /workspaces/DS_HS2026_LNW/DS_HS2026_LNW_I_Examples/AP01


### Datenbankverbindung zu PostgreSQL

In [2]:
# Zugangsdaten
user     = 'pgadmin'
password = 'geheim'
port     = '5432'
database = 'postgres'

# Host ermitteln: im Devcontainer heisst der PostgreSQL-Service 'db',
# bei Docker-in-Docker ist er unter 'localhost' erreichbar.
host = None
for candidate in ('db', 'localhost'):
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        host = candidate
        break
    except Exception:
        pass

# Connection URL für alle folgenden Abfragen
db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'

# Test der Datenbankverbindung
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Host:      ', host)
    print('Datenbank: ', connection.execute(text('SELECT current_database()')).scalar())
engine.dispose()

Host:       localhost
Datenbank:  postgres


### Daten nach PostgreSQL laden

In [3]:
# Tabellen aus der SQLite-Datei cinema.db nach PostgreSQL übertragen
tables = ['movies', 'cinemas', 'staff', 'screenings', 'customers', 'tickets']

sqlite_engine = create_engine('sqlite:///cinema.db')
engine = create_engine(db_connection_url)

for table in tables:
    df = pd.read_sql_query(f'SELECT * FROM {table}', con=sqlite_engine)
    df.to_sql(table, engine, if_exists='replace', index=False)
    print(f'{table:<12}{len(df):>5} Zeilen')

sqlite_engine.dispose()
engine.dispose()

movies         15 Zeilen
cinemas         6 Zeilen
staff          30 Zeilen
screenings    300 Zeilen
customers     505 Zeilen
tickets      2000 Zeilen


> **Hinweis — Backup mit SQLite**
>
> Falls die PostgreSQL-Verbindung nicht funktioniert, verwenden Sie für alle Aufgaben die
> SQLite-Datenbank `cinema.db`. Ersetzen Sie dazu in den Code-Zellen:
> ```python
> engine = create_engine(db_connection_url)
> ```
> durch:
>
> ```python
> engine = create_engine('sqlite:///cinema.db')
> ```
>
> Tabellen und Inhalte sind identisch, und alle Abfragen dieser Prüfung laufen unverändert
> in beiden Datenbanken.

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnt die bewertete Prüfung — 40 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Schema von `cinema.db` werden nicht gewertet.*

### A1 (4 Punkte)

Die Tabelle `screenings` enthält `base_price` und die Tabelle `tickets` enthält `paid_price`.

Ein Datenbankarchitekt sagt: *«`paid_price` ist redundant — der Preis steht bereits in `screenings.base_price`. Die Spalte sollte entfernt werden.»*

Hat er recht? Begründen Sie Ihre Antwort mit einem konkreten Szenario aus dem Kinobetrieb (z.B. Rabattaktionen, Gutscheine).

Nein, er hat nicht recht. Die Spalte ist nachwievor notwendig um zum Beispiel den erzielten Umsatz zu berechnen. Der PaidPrice kann vom BasePrice abweichen wenn z.B. Gutscheine eingelöst werden oder der Film an speziellen Tagen zu einem anderen Preis angeboten wird.

### A2 (4 Punkte)

Das Datenteam analysiert die Tabelle `customers` und findet folgende Werte in der Spalte `membership_type`:

```
Standard, standard, STANDARD, Standart, Premium, premium, Student, student, Senior
```

Beschreiben Sie das konkrete Datenproblem und nennen Sie die passende Data-Quality-Eigenschaft aus den Vorlesungsfolien (z.B. accurate, consistent, complete, reliable).

**Ihre Antwort:**

Die Daten sind nicht konsistent, da die Gross- und Kleinschreibung vorhanden ist und zusätzlich ein Schreibfehler enthalten ist (`Standart`).

### A3 (4 Punkte)

In `cinema.db` ist `paid_price` bei keinem Ticket höher als der `base_price` der zugehörigen Vorstellung. Angenommen, ein neues Ticket hätte `paid_price` = CHF 18.00 bei einem `base_price` von CHF 16.00: Wäre das **zwingend** ein Datenfehler? Begründen Sie in einem Satz.

**Ihre Antwort:**

Nein, es wäre kein Datenfehler. Es könnte ja auch sein, dass die Preise variieren oder der Preis von 18.00 CHF für ein VIP Ticket oder eine Kombi-Aktion mit einem Getränk ist.

---
## Teil B – SQL-Fehleranalyse (16 Punkte)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Geschäftsergebnis. Die tatsächlichen Ergebnisse aus `cinema.db` sind angegeben.*

### B1 (8 Punkte)

**Ziel:** Anzahl verkaufter Tickets pro Kino — für **alle** Kinos in der Datenbank.

```sql
SELECT c.name, COUNT(*) AS ticket_count
FROM screenings s
JOIN tickets t ON s.screeningid = t.screeningid
JOIN cinemas c ON s.cinemaid = c.cinemaid
GROUP BY c.name;
```

Die Tabelle `cinemas` enthält **6 Kinos**. Tatsächliches Ergebnis der SQL-Abfrage: nur **5 Zeilen**, Ticketanzahl zwischen 350 und 450.

Das Management fragt: *«Zeig mir auch Kinos, die noch keine Tickets verkauft haben.»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, sodass **alle 6 Kinos** erscheinen und ein Kino ohne Tickets korrekt mit `0` ausgewiesen wird. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

Der `JOIN` nimmt nur Werte welche in beiden Tabellen existieren, es muss `LEFT JOIN` verwendet werden um auch Kinos abzubilden, welche noch keine Tickets verkauft haben. Zusätzlich zählt `COUNT(*)` nur die Anzahl Zeilen anstatt die Anzahl der verkauften Tickets.

**Korrigierte SQL-Abfrage:**

In [10]:
engine = create_engine(db_connection_url)

df_B1 = pd.read_sql_query("""
SELECT c.name, COUNT(t.ticketid) AS ticket_count
FROM cinemas c
LEFT JOIN screenings s ON s.cinemaid = c.cinemaid
LEFT JOIN tickets t ON s.screeningid = t.screeningid
GROUP BY c.name;
""", con=engine)

engine.dispose()
df_B1

,name,ticket_count
0,Rex Kino,386
1,Cineplexx,383
2,Kinok,441
3,Pathé Küchlin,436
4,Kino Aurora,0
5,Arthouse Metropol,354


### B2 (8 Punkte)

**Ziel:** Liste aller Kunden (`customerid`, `firstname`, `lastname`), die noch nie ein Ticket gekauft haben.

```sql
SELECT c.customerid, c.firstname, c.lastname
FROM customers c
JOIN tickets t ON c.customerid = t.customerid
WHERE t.ticketid IS NULL;
```

Tatsächliches Ergebnis: 0 Zeilen — obwohl es Kunden ohne Tickets gibt.

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, die alle Kunden ohne Ticket ausgibt. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

Der `JOIN` war falsch, da customers welche noch nie ein Ticket gekauft haben auch nicht mit dessen id in der tickets Tabelle vorkommen.

**Korrigierte SQL-Abfrage:**

In [11]:
engine = create_engine(db_connection_url)

df_B2 = pd.read_sql_query("""
SELECT c.customerid, c.firstname, c.lastname
FROM customers c
LEFT JOIN tickets t ON c.customerid = t.customerid
WHERE t.ticketid IS NULL;
""", con=engine)

engine.dispose()
df_B2

,customerid,firstname,lastname
0,CU0503,Julia,Schmid
1,CU0505,Sandra,Fischer
2,CU0045,Sara,Schmidt
3,CU0504,Kevin,Keller
4,CU0432,Lucas,Wolf
5,CU0259,Max,Keller
6,CU0019,David,Krüger
7,CU0502,Thomas,Berger
8,CU0334,Mia,Krüger
9,CU0270,Marie,Braun


---
## Teil C – SQL-Abfragen schreiben (12 Punkte)

Schreiben Sie SQL-Abfragen auf `cinema.db` und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen. Worauf die Begründung eingehen soll, steht bei der Aufgabe.

### C1 (4 Punkte)

Die gültigen Werte in `customers.membership_type` sind `Standard`, `Premium`, `Student` und `Senior`. Schreiben Sie eine SQL-Abfrage, die alle Kunden (`customerid`, `firstname`, `lastname`, `membership_type`) mit einem ungültigen Wert in `membership_type` ausgibt — d.h. alle Werte ausser diesen vier. Begründen Sie Ihre Filterbedingung.

**Begründung der Vorgehensweise:**

Es wurde nach allen Werten gefiltert (`IN`) welche nicht (`NOT`) in der Liste (`('Standard', 'Premium', 'Student', 'Senior')`) vorkommen

**SQL-Abfrage:**

In [12]:
engine = create_engine(db_connection_url)

df_C1 = pd.read_sql_query("""
SELECT customerid, firstname, lastname, membership_type
FROM customers
WHERE membership_type NOT IN ('Standard', 'Premium', 'Student', 'Senior');
""", con=engine)

engine.dispose()
df_C1

,customerid,firstname,lastname,membership_type
0,CU0501,Anna,Müller,standard
1,CU0502,Thomas,Berger,STANDARD
2,CU0503,Julia,Schmid,Standart
3,CU0504,Kevin,Keller,premium
4,CU0505,Sandra,Fischer,student


### C2 (4 Punkte)

Wie hoch ist der durchschnittliche bezahlte Ticketpreis (`paid_price`) pro Mitgliedschaftstyp (`membership_type`)? Geben Sie `membership_type` und den Durchschnittspreis (`avg_price`) aus, gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`. Sortieren Sie absteigend nach `avg_price`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

Das Schlagwort 'pro' deutet auf ein `GROUP BY` statement hin. Die Daten müssen von den beiden Tabellen tickets und customers bezogen werden und mittels customerid gejoint werden. Das Sortieren wird mit `ORDER BY` gemacht.

**SQL-Abfrage:**

In [15]:
engine = create_engine(db_connection_url)

df_C2 = pd.read_sql_query("""
SELECT ROUND(CAST(AVG(t.paid_price) AS NUMERIC), 2) AS avg_price, c.membership_type
FROM tickets t
JOIN  customers c ON c.customerid = t.customerid
GROUP BY c.membership_type
ORDER BY avg_price DESC
""", con=engine)

engine.dispose()
df_C2

,avg_price,membership_type
0,15.62,Premium
1,15.59,Student
2,15.49,Standard
3,15.34,Senior


### C3 (4 Punkte)

Welche Kinos haben mehr als 400 Tickets verkauft? Geben Sie `name` und Anzahl Tickets (`ticket_count`) aus, absteigend sortiert nach `ticket_count`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

Es müssen die Tabellen cinema, screenings und tickets verbunden werden. Die Tabelle tickets ist ohne die Tabelle screenings für die Tabelle cinema nicht erreichbar.

**SQL-Abfrage:**

In [21]:
engine = create_engine(db_connection_url)

df_C3 = pd.read_sql_query("""
SELECT c.name, COUNT(t.ticketid) AS ticket_count
FROM cinemas c
JOIN screenings s ON c.cinemaid = s.cinemaid
JOIN tickets t ON t.screeningid = s.screeningid
GROUP BY c.cinemaid, c.name
HAVING COUNT(t.ticketid) > 400
ORDER BY ticket_count DESC
""", con=engine)

engine.dispose()
df_C3

,name,ticket_count
0,Kinok,441
1,Pathé Küchlin,436


---
## Zusätzliche Übungsteile

Die folgenden Aufgaben sind zusätzliche Prüfungstrainingsaufgaben. Verwenden Sie dieselbe Datenbank und dieselben Tabellen wie in den Teilen A–C oben.

## Zusätzlicher Teil A – Konzeptfragen (12 Punkte)

### A4 (4 Punkte)

Das Management möchte alle Kinos anzeigen, auch wenn dort noch keine Tickets verkauft wurden. Erklären Sie, weshalb dafür ein `LEFT JOIN` statt eines `INNER JOIN` geeignet ist. Beschreiben Sie ausserdem, welche Werte in den Spalten der rechten Tabelle bei einem Kino ohne Tickets erscheinen.

**Ihre Antwort:**

Ein `LEFT JOIN` ist notwendig, so dass alle Reihen der linken Tabelle übernommen werden und solche die auch in der rechten Tabelle vorkommen.

### A5 (4 Punkte)

Erklären Sie den Unterschied zwischen `WHERE` und `HAVING`. Verwenden Sie dabei ein Beispiel, bei dem nur Kinos mit mehr als 400 verkauften Tickets ausgegeben werden.

**Ihre Antwort:**

`WHERE` filtert **vor** der Gruppierung.  
`HAVING` filtert **nach** der Gruppierung.

### A6 (4 Punkte)

In einer Abfrage werden `customers` und `tickets` verbunden. Ein Kunde mit fünf gekauften Tickets erscheint danach fünfmal. Ist das zwingend ein Fehler? Begründen Sie Ihre Antwort mit der zugrunde liegenden 1:n-Beziehung und nennen Sie eine Möglichkeit, bei der Ausgabe wieder eine Zeile pro Kunde zu erhalten.

**Ihre Antwort:**

Nein, es ist kein Fehler, mit `DISTINCT` könnte dieses Problem behoben werden.

---
## Zusätzlicher Teil B – SQL-Fehleranalyse (16 Punkte)

*Jede Abfrage läuft syntaktisch durch, liefert aber nicht das gewünschte Ergebnis. Beschreiben Sie den Fehler und formulieren Sie die korrigierte Abfrage.*

### B3 (8 Punkte)

**Ziel:** Alle Kinos ausgeben, auch Kinos ohne Tickets. Die folgende Abfrage liefert jedoch nur Kinos, für die mindestens ein Ticket mit `paid_price > 15` existiert:

```sql
SELECT c.name, COUNT(t.ticketid) AS expensive_ticket_count
FROM cinemas c
LEFT JOIN screenings s ON c.cinemaid = s.cinemaid
LEFT JOIN tickets t ON t.screeningid = s.screeningid
WHERE t.paid_price > 15
GROUP BY c.cinemaid, c.name;
```

Erklären Sie, weshalb der `LEFT JOIN` hier seine Wirkung verliert. Korrigieren Sie die Abfrage so, dass alle Kinos erscheinen und nur Tickets mit `paid_price > 15` gezählt werden.

**Fehleranalyse und korrigierte Abfrage:**

Eine Bedingung auf der rechten Tabelle (tickets) in `WHERE` verwandelt einen `LEFT JOIN` in einen `INNER JOIN`.

In [24]:
engine = create_engine(db_connection_url)

df_B3 = pd.read_sql_query("""
SELECT
    c.name,
    COUNT(t.ticketid) AS expensive_ticket_count
FROM cinemas c
LEFT JOIN screenings s
    ON c.cinemaid = s.cinemaid
LEFT JOIN tickets t
    ON t.screeningid = s.screeningid
   AND t.paid_price > 15
GROUP BY c.cinemaid, c.name
ORDER BY expensive_ticket_count DESC;
""", con=engine)

engine.dispose()
df_B3

,name,expensive_ticket_count
0,Kinok,241
1,Pathé Küchlin,211
2,Arthouse Metropol,205
3,Rex Kino,181
4,Cineplexx,173
5,Kino Aurora,0


### B4 (8 Punkte)

**Ziel:** Anzahl der Kunden pro Mitgliedschaftstyp. Die folgende Abfrage soll nur Mitgliedschaftstypen mit mindestens 100 Kunden anzeigen:

```sql
SELECT membership_type, COUNT(*) AS customer_count
FROM customers
WHERE COUNT(*) >= 100
GROUP BY membership_type;
```

Die Abfrage läuft nicht korrekt. Erklären Sie, weshalb `COUNT(*)` nicht in `WHERE` verwendet werden darf, und schreiben Sie die korrigierte Abfrage. Sortieren Sie absteigend nach `customer_count`.

**Fehleranalyse und korrigierte Abfrage:**

WHERE wird vor der Gruppierung und Aggregation ausgeführt. Zu diesem Zeitpunkt existiert COUNT(*) pro Mitgliedschaftstyp noch nicht.

In [27]:
engine = create_engine(db_connection_url)

df_B4 = pd.read_sql_query("""
SELECT membership_type, COUNT(customerid) AS customer_count
FROM customers
GROUP BY membership_type
HAVING COUNT(customerid) >= 100
ORDER BY customer_count DESC;
""", con=engine)

engine.dispose()
df_B4

,membership_type,customer_count
0,Student,135
1,Senior,134
2,Premium,128
3,Standard,103


---
## Zusätzlicher Teil C – SQL-Abfragen schreiben (16 Punkte)

### C4 (4 Punkte)

Geben Sie alle Filme (`movieid`, `title`, `duration_min`) aus, deren Laufzeit zwischen 90 und 120 Minuten liegt. Sortieren Sie aufsteigend nach `duration_min` und bei gleicher Laufzeit alphabetisch nach `title`.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [33]:
engine = create_engine(db_connection_url)

df_C4 = pd.read_sql_query("""
SELECT movieid, title, duration_min
FROM movies
WHERE duration_min BETWEEN 90 AND 120
ORDER BY duration_min DESC, title ASC;
""", con=engine)

engine.dispose()
df_C4

,movieid,title,duration_min
0,M007,Barbie,114
1,M014,Past Lives,106


### C5 (4 Punkte)

Geben Sie für jeden Film (`movieid`, `title`) die Anzahl der Vorstellungen (`screening_count`) aus. Berücksichtigen Sie auch Filme ohne Vorstellung. Sortieren Sie absteigend nach `screening_count` und danach aufsteigend nach `title`.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [38]:
engine = create_engine(db_connection_url)

df_C5 = pd.read_sql_query("""
SELECT m.movieid, m.title, COUNT(s.screeningid) AS screening_count
FROM movies m
LEFT JOIN screenings s ON s.movieid = m.movieid
GROUP BY m.movieid, m.title
ORDER BY screening_count DESC, m.title ASC;
""", con=engine)

engine.dispose()
df_C5

,movieid,title,screening_count
0,M006,Oppenheimer,27
1,M003,Interstellar,26
2,M005,Everything Everywhere,24
3,M002,The Dark Knight,24
4,M008,The Godfather,23
5,M009,Pulp Fiction,22
6,M013,Poor Things,21
7,M007,Barbie,20
8,M004,Parasite,20
9,M014,Past Lives,19


### C6 (4 Punkte)

Geben Sie `firstname`, `lastname` und `membership_type` aller Kunden aus, die mindestens ein Ticket gekauft haben, aber noch nie ein Ticket mit `paid_price < 12` gekauft haben. Jeder Kunde soll höchstens einmal erscheinen. Sortieren Sie nach `lastname` und `firstname`.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [57]:
engine = create_engine(db_connection_url)

df_C6 = pd.read_sql_query("""
SELECT
    c.firstname,
    c.lastname,
    c.membership_type
FROM customers c
JOIN tickets t
    ON t.customerid = c.customerid
GROUP BY
    c.customerid,
    c.firstname,
    c.lastname,
    c.membership_type
HAVING MIN(t.paid_price) >= 12
ORDER BY c.firstname, c.lastname;
""", con=engine)

engine.dispose()
df_C6

,firstname,lastname,membership_type
0,Anna,Bauer,Student
1,Anna,Hartmann,Senior
2,Anna,Hartmann,Premium
3,Anna,Hartmann,Standard
4,Anna,Huber,Premium
...,...,...,...
297,Tom,Weber,Student
298,Tom,Wolf,Premium
299,Tom,Wolf,Premium
300,Tom,Zimmermann,Standard


### C7 (4 Punkte)

Berechnen Sie für jedes Kino den gesamten Umsatz aus verkauften Tickets. Geben Sie `name` und den auf 2 Dezimalstellen gerundeten Umsatz (`total_revenue`) aus. Verwenden Sie dafür den tatsächlich bezahlten Preis `paid_price`, berücksichtigen Sie nur Kinos mit mindestens einem Ticket und sortieren Sie absteigend nach `total_revenue`.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [61]:
engine = create_engine(db_connection_url)

df_C7 = pd.read_sql_query("""
SELECT c.name, ROUND(CAST(SUM(t.paid_price) AS NUMERIC), 2) AS total_revenue
FROM cinemas c
JOIN screenings s ON s.cinemaid = c.cinemaid
JOIN tickets t ON t.screeningid = s.screeningid
GROUP BY c.cinemaid, c.name
ORDER BY total_revenue DESC
;""", con=engine)

engine.dispose()
df_C7

,name,total_revenue
0,Kinok,6975.89
1,Pathé Küchlin,6731.89
2,Rex Kino,5858.15
3,Cineplexx,5855.13
4,Arthouse Metropol,5595.37


---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os, platform, socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('IP Address:', socket.gethostbyname(socket.gethostname()))
print('-----------------------------------')